# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebakk/flyrank-ml-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "Scoring/Probability Estimation"
print(f"Configured Task Pipeline: {task_type}")



Configured Task Pipeline: Scoring/Probability Estimation


This task is framed as a **Scoring / Probability Estimation** problem. Instead of simple classification, we want to predict a continuous probability score (between 0.0 and 1.0) that a traveler will click or select a specific flight itinerary. This continuous score allows our downstream system to dynamically sort and rank the best flight options for the user from top to bottom.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sketching the binary target distribution
import numpy as np
mock_target = np.array([1, 0, 0, 1, 0, 0])
print(f"Target variable values distribution template: {mock_target}")



Target variable values distribution template: [1 0 0 1 0 0]


The target variable is an **observed outcome**: `is_clicked` (or `is_selected`), represented as a binary indicator where 1 means the user clicked the flight and 0 means they ignored it. This label comes directly from historical clickstream infrastructure and server logs, capturing real human decisions rather than an artificial, pre-defined rule.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setting the defensible threshold metric
target_roc_auc = 0.78
print(f"Defensible Success Metric Threshold: ROC-AUC >= {target_roc_auc}")



Defensible Success Metric Threshold: ROC-AUC >= 0.78


The success metric to defend is **ROC-AUC (Receiver Operating Characteristic - Area Under the Curve)**. A baseline score is considered 'good' if it is **≥ 0.78**. ROC-AUC is ideal here because it evaluates the model's ability to cleanly separate selected flights from ignored ones by score magnitude, handling the heavy class imbalance (since users ignore most flights) perfectly.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# Try to load the real data slice, otherwise fallback to schema preview
file_path = '../data/flyrank_slice.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    # Exact structural blueprint representation
    df = pd.DataFrame([{
        "search_id": "sess_88492",
        "itinerary_id": "iti_car_441",
        "price_usd": 349.50,
        "duration_minutes": 210,
        "stops": 1,
        "is_clicked": 1
    }, {
        "search_id": "sess_88492",
        "itinerary_id": "iti_car_992",
        "price_usd": 520.00,
        "duration_minutes": 140,
        "stops": 0,
        "is_clicked": 0
    }])

print(f"Unit of Analysis: One row = One itinerary option inside a user search session.")
print(f"Dataframe Dimensions: {df.shape}\n")
print(df.head())



Unit of Analysis: One row = One itinerary option inside a user search session.
Dataframe Dimensions: (2, 6)

    search_id itinerary_id  price_usd  duration_minutes  stops  is_clicked
0  sess_88492  iti_car_441      349.5               210      1           1
1  sess_88492  iti_car_992      520.0               140      0           0


The unit of analysis is: **One row = One unique itinerary option presented within a specific user search session.** If a user searches for a flight and sees 10 options, that single search session will generate 10 distinct rows in our dataframe, each containing specific parameters for that flight option.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple function proving rule fragility across user personas
def fixed_heuristic(price, stops):
    return 1 if price < 400 and stops == 0 else 0

print("Leisure trip ($350, 1 stop): Passed to heuristic ->", fixed_heuristic(350, 1), "(Should be scored high)")
print("Business trip ($600, 0 stops): Passed to heuristic ->", fixed_heuristic(600, 0), "(Should be scored high)")



Leisure trip ($350, 1 stop): Passed to heuristic -> 0 (Should be scored high)
Business trip ($600, 0 stops): Passed to heuristic -> 0 (Should be scored high)


A fixed rule like `if price < 400 and stops == 0` completely fails because user selection behavior is highly non-linear and context-dependent. A business traveler will readily accept a $700 ticket for a non-stop morning flight, while a leisure student traveler will prefer a $300 ticket with two layovers. Hardcoded rules cannot scale to balance shifting combinations of price, duration, baggage rules, and seasonal demand simultaneously. ML dynamically weights these multi-variable trade-offs where human heuristics break down.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.